Doing the required imports

In [1]:
import datetime
import os
import shutil
import sys

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../data"))

import dataset

os.chdir("..")

Define the constants

In [2]:
SPLITS = 7
IMAGE_SIZE = 640

Creating the data directories

In [3]:
dataset.create_dataset_directories()

Get the data folds

In [4]:
(
    labels,
    classes,
    kfolds,
    labels_df,
    labels_df_with_counts,
    cls_idx,
    folds_df,
    valid_labels_df,
) = dataset.get_data_folds()

Create the yaml directories

In [5]:
images, save_path, ds_yamls = dataset.create_yml_directories(folds_df, classes)

Copy validation data

In [6]:
dataset.copy_validation_data(images, labels, valid_labels_df, save_path)

Fucntion to copy images

In [7]:
def copy_images_and_labels(
    images, labels, folds_df, save_path, k
):
    for image, label in zip(images, labels):
        if image.stem not in folds_df.index:
            continue
        split = f"split_{k + 1}"
        k_split = folds_df.loc[image.stem][split]
        # Destination directory
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"
        image_name = f"{image.stem}{image.suffix}"
        label_name = f"{label.stem}{label.suffix}"
        img = dataset.load_image(image)
        img.save(img_to_path / image_name)
        shutil.copy(label, lbl_to_path / label_name)

Model training

In [8]:
from ultralytics import YOLO

In [9]:
def train_models(ds_yamls, images, labels, folds_df, save_path):
    results = {}

    # Define your additional arguments here
    folds = SPLITS
    batch = 16
    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    project = f"runs/train/{folds}fold/{current_time}"
    epochs = 40

    for k in range(folds):
        copy_images_and_labels(images, labels, folds_df, save_path, k)
        dataset_yaml = ds_yamls[k]
        model = YOLO("yolo11s.pt", task="detect")
        model.train(
            data=dataset_yaml,
            epochs=epochs,
            batch=batch,
            imgsz=IMAGE_SIZE,
            project=project,
            patience=10,
            seed=0,
            optimizer="AdamW",
            lr0=0.0005,
            momentum=0.9,
            flipud=0.5,
            crop_fraction=0.9,
            mixup=0.1,
            cls=2,
        )
        results[k] = model.metrics

        model.val(
            data=dataset_yaml,
            project=f"{project}/validation",
            imgsz=IMAGE_SIZE,
        )

        shutil.rmtree(f"{save_path}/split_{k + 1}")

    return project

In [ ]:
train_models(ds_yamls, images, labels, folds_df, save_path)